# Load Library

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
import pandas as pd
from sklearn.metrics import (accuracy_score,
    classification_report, confusion_matrix)

In [ ]:
import pandas as pd
from tqdm.notebook import tqdm

In [ ]:
!nvidia-smi

### SETUP Project Path

In [ ]:
PROJECT_PATH = '/project/lt200393-foullm/sam/2026/workshop/llm-finetune-workshop'

# Load Test set

In [ ]:
data_path = f"{PROJECT_PATH}/dataset/format_dataset/wisesight_process/wisesight_sentiment_test.json"

In [ ]:
test_df = pd.read_json(data_path, lines=True)

In [ ]:
test_df

In [ ]:
test_df['messages'][0]

# Load Baseline Model

In [ ]:
model_name = f"{PROJECT_PATH}/model/qwen3-4b"

In [ ]:
base_tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

In [ ]:
base_model

# Load Trained Model

In [ ]:
model_name = f"{PROJECT_PATH}/trained_model/qwen3-4b-sentiment"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

In [ ]:
model

## Check GPU Usage

In [ ]:
!nvidia-smi

In [ ]:
SYSTEM_PROMPT = """You are an AI assistant specialized in social sentiment analysis.
Your task is to classify the sentiment expressed in a given piece of text from social media into one of the following categories:

positive: The text expresses favorable, supportive, or happy emotions.
negative: The text expresses unfavorable, critical, or unhappy emotions.
neutral: The text expresses neither positive nor negative sentiment, or it is objective, factual, or ambiguous.
"""

In [ ]:
print(SYSTEM_PROMPT)

# Create Inference Function

In [ ]:
def inference(prompt, model=model, tokenizer=tokenizer):
    messages = [
    {"role" : "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": prompt+'/no_think'}
    ]  
    
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True # Switches between thinking and non-thinking modes. Default is True.
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    # conduct text completion
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=100,
        temperature=0.001
    )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
    
    # parsing thinking content
    try:
        # rindex finding 151668 (</think>)
        index = len(output_ids) - output_ids[::-1].index(151668)
    except ValueError:
        index = 0
    
    thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
    content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")
    
    return content.strip()

# Inference Baseline Model

In [ ]:
def report(actual_y, predict_y):
    labels = ["negative", "neutral", "positive"]
    accuracy = accuracy_score(y_true=actual_y, y_pred=predict_y)
    print(f'Accuracy:\n{accuracy:.3f}')
    class_report = classification_report(y_true=actual_y, y_pred=predict_y)
    print(f'\nClassification Report:\n{class_report}')
    conf_matrix = confusion_matrix(y_true=actual_y, y_pred=predict_y, labels=labels)
    print(f'\nConfusion Matrix:\n{conf_matrix}')

In [ ]:
predict_y = []
actual_y = []

limits = 100
counter = 0
for row in tqdm(test_df.iterrows()):
    predict_y.append(inference(row[1][0][1]['content'], base_model, base_tokenizer))
    actual_y.append(row[1][0][2]['content'])
    counter += 1
    if counter == limits:
        break

In [ ]:
predict_y[:10]

In [ ]:
actual_y[:10]

In [ ]:
post_predict_y = []
for row in predict_y:
    if 'neutral' in row.lower():
        post_predict_y.append('neutral')
    elif 'positive' in row.lower():
        post_predict_y.append('positive')
    else:
        post_predict_y.append('negative')

In [ ]:
post_predict_y[:10]

In [ ]:
report(actual_y, post_predict_y)

# Inference Trained Model

In [ ]:
predict_y = []
actual_y = []

limits = 100
counter = 0
for row in tqdm(test_df.iterrows()):
    predict_y.append(inference(row[1][0][1]['content']))
    actual_y.append(row[1][0][2]['content'])
    counter += 1
    if counter == limits:
        break

In [ ]:
report(actual_y, predict_y)

# Evaluate Trained Model Performance

In [ ]:
predict_y = []
actual_y = []

for row in tqdm(test_df.iterrows()):
    predict_y.append(inference(row[1][0][1]['content']))
    actual_y.append(row[1][0][2]['content'])

In [ ]:
report(actual_y, predict_y)